# Book Recommendations with Book-Crossing

*Item-Based Collaborative Filtering with a Bronze–Silver–Gold Data Pipeline*

We want to see if the ratings people gave to books could help us suggest what they might like next. I start with the raw files, clean them, and use the ratings that are left to make recommendations.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import io
from html import unescape

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, load_npz, save_npz
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

pd.set_option("display.max_colwidth", None)

zip_path = Path("book-crossings.zip")
if not zip_path.exists():
    from google.colab import files
    files.upload()
if not zip_path.exists():
    raise FileNotFoundError("Upload the file named book-crossings.zip.")

for folder in ["bronze", "silver", "gold"]:
    Path(folder).mkdir(exist_ok=True)

Saving book-crossings.zip to book-crossings.zip


## 1. Bronze

I save a copy of each raw table first so I can go back and check what was in the original data. I read the files with Latin-1 encoding. Some book titles contain semicolons, so for Books I split at the boundary between quoted fields (`";"`). I add the time I loaded the files and save them as Parquet.

In [2]:
source_files = {
    "users": "BX-Users.csv",
    "books": "BX-Books.csv",
    "ratings": "BX-Book-Ratings.csv",
}

timestamp = pd.Timestamp.now(tz="UTC")
with ZipFile(zip_path) as archive:
    for table, filename in source_files.items():
        with io.TextIOWrapper(archive.open(filename), encoding="latin-1") as source:
            if table == "books":
                rows = [line.rstrip("\r\n")[1:-1].split('";"') for line in source]
                header, *records = rows
                assert all(len(row) == len(header) for row in records)
                frame = pd.DataFrame(records, columns=header, dtype="string")
            else:
                frame = pd.read_csv(source, sep=";", keep_default_na=False,
                                    dtype={"ISBN": "string"})
        frame["ingestion_timestamp"] = timestamp
        frame.to_parquet(f"bronze/{table}.parquet", index=False)
        print(table, frame.shape)

users (278858, 4)
books (271379, 9)
ratings (1149780, 4)


## 2. Bronze exploration

In [3]:
raw_users = pd.read_parquet("bronze/users.parquet")
raw_books = pd.read_parquet("bronze/books.parquet")
raw_ratings = pd.read_parquet("bronze/ratings.parquet")

### Check the raw tables

In [4]:
for name, frame in [("Users", raw_users), ("Books", raw_books), ("Ratings", raw_ratings)]:
    missing = frame.isna() | frame.isin(["", "NULL"])
    print(name, frame.shape)
    display(frame.head(3))
    display(pd.DataFrame({"dtype": frame.dtypes.astype(str), "missing": missing.sum()}))

Users (278858, 4)


,User-ID,Location,Age,ingestion_timestamp
0,1,"nyc, new york, usa",NULL,2026-11-27 13:35:34.152299+00:00
1,2,"stockton, california, usa",18,2026-11-27 13:35:34.152299+00:00
2,3,"moscow, yukon territory, russia",NULL,2026-11-27 13:35:34.152299+00:00


,dtype,missing
User-ID,int64,0
Location,object,0
Age,object,110762
ingestion_timestamp,"datetime64[us, UTC]",0


Books (271379, 9)


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,ingestion_timestamp
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.01.THUMBZZZ.jpg,http://images.amazon.com/images/P/0195153448.01.MZZZZZZZ.jpg,http://images.amazon.com/images/P/0195153448.01.LZZZZZZZ.jpg,2026-11-27 13:35:34.152299+00:00
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.01.THUMBZZZ.jpg,http://images.amazon.com/images/P/0002005018.01.MZZZZZZZ.jpg,http://images.amazon.com/images/P/0002005018.01.LZZZZZZZ.jpg,2026-11-27 13:35:34.152299+00:00
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.01.THUMBZZZ.jpg,http://images.amazon.com/images/P/0060973129.01.MZZZZZZZ.jpg,http://images.amazon.com/images/P/0060973129.01.LZZZZZZZ.jpg,2026-11-27 13:35:34.152299+00:00


,dtype,missing
ISBN,string,0
Book-Title,string,0
Book-Author,string,0
Year-Of-Publication,string,0
Publisher,string,0
Image-URL-S,string,0
Image-URL-M,string,0
Image-URL-L,string,0
ingestion_timestamp,"datetime64[us, UTC]",0


Ratings (1149780, 4)


,User-ID,ISBN,Book-Rating,ingestion_timestamp
0,276725,034545104X,0,2026-11-27 13:35:34.152299+00:00
1,276726,0155061224,5,2026-11-27 13:35:34.152299+00:00
2,276727,0446520802,0,2026-11-27 13:35:34.152299+00:00


,dtype,missing
User-ID,int64,0
ISBN,string,0
Book-Rating,int64,0
ingestion_timestamp,"datetime64[us, UTC]",0


### Check ratings

In [5]:
print("Rating distribution")
display(raw_ratings["Book-Rating"].value_counts().sort_index().rename("count").to_frame())

Rating distribution


,count
Book-Rating,
0,716109
1,1770
2,2759
3,5996
4,8904
5,50974
6,36924
7,76457
8,103736


### Check ages and publication years

In [6]:
current_year = pd.Timestamp.today().year
ages = pd.to_numeric(raw_users["Age"], errors="coerce")
years = pd.to_numeric(raw_books["Year-Of-Publication"], errors="coerce")
print("Age and publication year distributions")
display(pd.DataFrame({"age": ages.describe(), "year": years.describe()}))
age_bins = [-np.inf, 4, 17, 29, 44, 64, 100, np.inf]
year_bins = [-np.inf, 0, 1949, 1969, 1989, 1999, current_year, np.inf]
display(pd.cut(ages, age_bins).value_counts(sort=False).rename("user count").to_frame())
display(pd.cut(years, year_bins).value_counts(sort=False).rename("book count").to_frame())
print("Ages outside 5–100:", int(((ages < 5) | (ages > 100)).sum()))
print("Zero or future years:", int(((years == 0) | (years > current_year)).sum()))

Age and publication year distributions


,age,year
count,168096.000000,271379.0
mean,34.751434,1959.75605
std,14.428097,258.011363
min,0.000000,0.0
25%,24.000000,1989.0
50%,32.000000,1995.0
75%,44.000000,2000.0
max,244.000000,2050.0


,user count
Age,
"(-inf, 4.0]",882
"(4.0, 17.0]",11428
"(17.0, 29.0]",59192
"(29.0, 44.0]",56269
"(44.0, 64.0]",35645
"(64.0, 100.0]",4314
"(100.0, inf]",366


,book count
Year-Of-Publication,
"(-inf, 0.0]",4619
"(0.0, 1949.0]",293
"(1949.0, 1969.0]",2396
"(1969.0, 1989.0]",65554
"(1989.0, 1999.0]",126024
"(1999.0, 2026.0]",72482
"(2026.0, inf]",11


Ages outside 5–100: 1248
Zero or future years: 4630


### Check activity and sparsity

In [7]:
print("Rating records per user and per ISBN with at least one record")
counts = {"per user": raw_ratings.groupby("User-ID").size(),
          "per ISBN": raw_ratings.groupby("ISBN").size()}
display(pd.DataFrame({name: values.describe() for name, values in counts.items()}))

# Use catalog users and books as the denominator, and count distinct pairs in that catalog.
in_catalog = raw_ratings["User-ID"].isin(raw_users["User-ID"]) & raw_ratings["ISBN"].isin(raw_books["ISBN"])
pairs = raw_ratings.loc[in_catalog, ["User-ID", "ISBN"]].drop_duplicates()
coverage = 100 * len(pairs) / (raw_users["User-ID"].nunique() * raw_books["ISBN"].nunique())
print(f"Recorded pairs: {coverage:.6f}% (empty pairs: {100 - coverage:.6f}%)")

Rating records per user and per ISBN with at least one record


,per user,per ISBN
count,105283.000000,340556.000000
mean,10.920851,3.376185
std,90.562825,12.436252
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,4.000000,2.000000
max,13602.000000,2502.000000


Recorded pairs: 0.001363% (empty pairs: 99.998637%)


# My Observations :-

- **Tables and missing values.** I found 278,858 users, 271,379 books, and 1,149,780 rating records. Age is stored as text and 110,762 ages are missing, so I'll convert ages to numbers and fill the missing ones with the median valid age.
- **Ratings.** There are 716,109 zeros. A zero means the user didn't give an explicit rating, so I'll keep ratings from 1 to 10 for the model.
- **Ages and years.** I found 1,248 ages outside 5–100 and 4,630 years that were zero or in the future. I'll count those values as missing in Silver.
- **Activity.** The median is just one rating record per active user and per rated ISBN. I'll try a few minimum rating counts and see how much data is left.
- **Sparsity.** About 0.001363% of possible user–book pairs in the catalog have a rating record. I'll filter out low-activity users and books in Silver, then use a sparse matrix in Gold.

## 3. Silver

Here I clean each table and look at a few rows again to see what changed. I also check the counts before and after filtering.

In [8]:
users = raw_users.drop(columns="ingestion_timestamp").drop_duplicates().copy()
books = raw_books.drop(columns="ingestion_timestamp").drop_duplicates().copy()
ratings = raw_ratings.drop(columns="ingestion_timestamp").drop_duplicates().copy()

### Clean users

In [9]:
display(users[["User-ID", "Location", "Age"]].head())
age = pd.to_numeric(users["Age"], errors="coerce")
age = age.where(age.between(5, 100))
users["Age"] = age.fillna(age.median())
location = users.pop("Location").str.split(",", n=2, expand=True)
location.columns = ["City", "State", "Country"]
for column in location:
    users[column] = location[column].str.strip().str.lower().replace("", pd.NA)

display(users.head())
print("Missing ages before:", int(age.isna().sum()))
print("Missing ages after:", int(users["Age"].isna().sum()))

,User-ID,Location,Age
0,1,"nyc, new york, usa",NULL
1,2,"stockton, california, usa",18
2,3,"moscow, yukon territory, russia",NULL
3,4,"porto, v.n.gaia, portugal",17
4,5,"farnborough, hants, united kingdom",NULL


,User-ID,Age,City,State,Country
0,1,32.0,nyc,new york,usa
1,2,18.0,stockton,california,usa
2,3,32.0,moscow,yukon territory,russia
3,4,17.0,porto,v.n.gaia,portugal
4,5,32.0,farnborough,hants,united kingdom


Missing ages before: 112010
Missing ages after: 0


### Clean books

In [10]:
year = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
books["Year-Of-Publication"] = year.where(year.ne(0) & year.le(current_year)).astype("Int64")
books = books.drop(columns=["Image-URL-S", "Image-URL-M", "Image-URL-L"])
for column in ["Book-Title", "Book-Author", "Publisher"]:
    cleaned = books[column].map(unescape).str.replace(r"\s+", " ", regex=True).str.strip()
    books[column] = (cleaned.str.title()
                     .str.replace(r"(?<=\w)'S", "'s", regex=True)
                     .replace(["", "Null"], pd.NA))
books["ISBN"] = books["ISBN"].str.strip().str.upper()
display(books.head())
print("Missing publication years after cleaning:", int(books["Year-Of-Publication"].isna().sum()))

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,Harperflamingo Canada
2,0060973129,Decision In Normandy,Carlo D'Este,1991,Harperperennial
3,0374157065,Flu: The Story Of The Great Influenza Pandemic Of 1918 And The Search For The Virus That Caused It,Gina Bari Kolata,1999,Farrar Straus Giroux
4,0393045218,The Mummies Of Urumchi,E. J. W. Barber,1999,W. W. Norton & Company


Missing publication years after cleaning: 4630


### Clean ratings

In [11]:
print("Raw rating records:", len(raw_ratings))
print("Raw zero ratings:", int(raw_ratings["Book-Rating"].eq(0).sum()))
ratings["ISBN"] = ratings["ISBN"].str.strip().str.upper()

users = users.drop_duplicates(subset="User-ID")
books = books.drop_duplicates(subset="ISBN")
ratings = ratings.loc[
    ratings["Book-Rating"].between(1, 10)
    & ratings["User-ID"].isin(users["User-ID"])
    & ratings["ISBN"].isin(books["ISBN"])
]
# Keep one value per user–book pair if any ratings disagree.
ratings = ratings.groupby(["User-ID", "ISBN"], as_index=False)["Book-Rating"].mean()
print("Valid explicit ratings before the activity cutoff:", len(ratings))
display(ratings.head())

Raw rating records: 1149780
Raw zero ratings: 716109
Valid explicit ratings before the activity cutoff: 383853


,User-ID,ISBN,Book-Rating
0,8,0002005018,5.0
1,8,074322678X,5.0
2,8,0887841740,5.0
3,8,1552041778,5.0
4,8,1567407781,6.0


### Compare minimum rating counts

I check the user and book counts more than once. If I drop a book with too few ratings, some users might fall below the cutoff too.

In [12]:
def filter_ratings(frame, minimum):
    while True:
        user_count = frame.groupby("User-ID")["ISBN"].transform("size")
        book_count = frame.groupby("ISBN")["User-ID"].transform("size")
        kept = frame.loc[user_count.ge(minimum) & book_count.ge(minimum)]
        if len(kept) == len(frame):
            return kept
        frame = kept

cutoffs = {minimum: filter_ratings(ratings, minimum) for minimum in [5, 10, 20]}
summary = pd.DataFrame([
    {"minimum": minimum, "users": data["User-ID"].nunique(),
     "books": data["ISBN"].nunique(), "ratings": len(data)}
    for minimum, data in cutoffs.items()
])
display(summary)

,minimum,users,books,ratings
0,5,6858,9092,115332
1,10,1820,2030,41469
2,20,0,0,0


I went with **10 explicit ratings per user and per book**. With 5, more books stay in the model, but some have only five ratings to compare. With 20, there was nothing left after filtering. Using 10 gave me 1,820 users, 2,030 books, and 41,469 ratings.

In [13]:
minimum = 10
ratings = cutoffs[minimum]
users = users.loc[users["User-ID"].isin(ratings["User-ID"])]
books = books.loc[books["ISBN"].isin(ratings["ISBN"])]
assert ratings.groupby("User-ID").size().min() >= minimum
assert ratings.groupby("ISBN").size().min() >= minimum
print("Lowest rating count per user:", ratings.groupby("User-ID").size().min())
print("Lowest rating count per book:", ratings.groupby("ISBN").size().min())

for name, frame in [("users", users), ("books", books), ("ratings", ratings)]:
    frame.to_parquet(f"silver/{name}.parquet", index=False)
    print(name, frame.shape)

Lowest rating count per user: 10
Lowest rating count per book: 10
users (1820, 5)
books (2030, 5)
ratings (41469, 3)


## 4. Gold

I put users on the rows and books on the columns. Most pairs have no rating, so the sparse matrix just stores the ratings that exist. I also save the book titles and authors and the rating counts and averages.

In [14]:
ratings = pd.read_parquet("silver/ratings.parquet")
books = pd.read_parquet("silver/books.parquet")

user_ids = pd.Index(sorted(ratings["User-ID"].unique()), name="User-ID")
book_ids = pd.Index(sorted(ratings["ISBN"].unique()), name="ISBN")
matrix = csr_matrix(
    (ratings["Book-Rating"], (user_ids.get_indexer(ratings["User-ID"]),
                              book_ids.get_indexer(ratings["ISBN"]))),
    shape=(len(user_ids), len(book_ids)),
)
save_npz("gold/user_book_ratings.npz", matrix)
user_ids.to_frame(index=False).to_parquet("gold/user_ids.parquet", index=False)
lookup = books.set_index("ISBN").loc[book_ids, ["Book-Title", "Book-Author"]].reset_index()
lookup.to_parquet("gold/book_lookup.parquet", index=False)
assert lookup["ISBN"].tolist() == book_ids.tolist()
assert matrix.nnz == len(ratings)

for name, key in [("user", "User-ID"), ("book", "ISBN")]:
    stats = ratings.groupby(key)["Book-Rating"].agg(average_rating="mean", rating_count="size")
    stats.reset_index().to_parquet(f"gold/{name}_stats.parquet", index=False)

print("User–book matrix:", matrix.shape, "| Recorded ratings:", matrix.nnz)
display(lookup.head())

User–book matrix: (1820, 2030) | Recorded ratings: 41469


,ISBN,Book-Title,Book-Author
0,0020199600,Great Gatsby (Reissue),F. Scott Fitzgerald
1,0020442203,"Lion, The Witch And The Wardrobe",C.S. Lewis
2,002542730X,Politically Correct Bedtime Stories: Modern Tales For Our Life And Times,James Finn Garner
3,0028604199,Once Upon A More Enlightened Time: More Politically Correct Bedtime Stories,James Finn Garner
4,0060002050,On A Wicked Dawn (Cynster Novels),Stephanie Laurens


## 5. Item-based collaborative filtering

I turn the matrix around so I can compare books row by row. I use cosine similarity to see how similar their rating patterns are.

In [15]:
user_book = load_npz("gold/user_book_ratings.npz").tocsr()
book_user = user_book.T.tocsr()
user_ids = pd.Index(pd.read_parquet("gold/user_ids.parquet")["User-ID"])
lookup = pd.read_parquet("gold/book_lookup.parquet")
titles = lookup["Book-Title"].fillna(lookup["ISBN"])
book_counts = book_user.getnnz(axis=1)

history = raw_ratings.loc[raw_ratings["User-ID"].isin(user_ids), ["User-ID", "ISBN"]].copy()
history["ISBN"] = history["ISBN"].str.strip().str.upper()
seen = history.groupby("User-ID")["ISBN"].agg(set).to_dict()

### Find similar books

I look up the title and return the 10 other books with the highest similarity scores. If a title has several editions, I use the one with the most ratings in Silver.

In [16]:
def neighbors(book_index, count):
    # Compare this book with every book using their user-rating vectors.
    similarities = cosine_similarity(book_user[book_index], book_user)[0]
    similarities[book_index] = -1  # Leave out the original book.
    nearest = np.argsort(-similarities)[:count]
    return [(index, similarities[index]) for index in nearest]

def similar_books(title):
    matches = np.flatnonzero(titles.str.casefold().eq(title.strip().casefold()))
    if len(matches) == 0:
        return []
    book_index = matches[book_counts[matches].argmax()]
    return [titles.iloc[i] for i, _ in neighbors(book_index, 10)]

### Recommend books to a user

I count a book as a favorite if the user rated it 8 or higher. For each favorite, I look at its 30 nearest books and score the ones with positive similarity. For example, if a favorite was rated 9 and another book has a similarity of 0.4, that book gets 3.6 points. If two favorites lead to the same book, I add the points. I skip books the user already interacted with, including zero-rating entries.

In [17]:
def recommend(user_id, n=10):
    if user_id not in user_ids or n <= 0:
        return []
    row = user_book[user_ids.get_loc(user_id)]
    already_seen = seen.get(user_id, set())
    seen_titles = set(titles[lookup["ISBN"].isin(already_seen)].str.casefold())
    scores = {}
    book_isbns = lookup["ISBN"]
    for book_index, rating in zip(row.indices, row.data):
        if rating < 8:
            continue
        for candidate, similarity in neighbors(book_index, 30):
            title = titles.iloc[candidate]
            if similarity <= 0 or book_isbns.iloc[candidate] in already_seen:
                continue
            if title.casefold() not in seen_titles:
                scores[title] = scores.get(title, 0) + rating * similarity
    return sorted(scores, key=lambda title: (-scores[title], title))[:n]

## 6. Demo and sanity check

I put the favorites on the left and the recommendations on the right. Each recommendation comes from the full set of favorites. Some rows on the left are blank because those users had fewer than 10 favorites to list.

In [18]:
example_title = titles.iloc[book_counts.argmax()]
print("10 books similar to", example_title)
display(pd.DataFrame({"Similar book": similar_books(example_title)}))
assert len(similar_books(example_title)) == 10

10 books similar to The Lovely Bones: A Novel


,Similar book
0,Where The Heart Is (Oprah'S Book Club (Paperback))
1,Lucky : A Memoir
2,The Da Vinci Code
3,The Pilot'S Wife : A Novel
4,Good In Bed
5,Nights In Rodanthe
6,The Pact: A Love Story
7,The Lake House
8,Lucky
9,White Oleander : A Novel (Oprah'S Book Club)


### Try the recommender on three users
Pick three users whose complete lists of favorites fit in the demo table.

In [20]:
favorite_counts = (user_book >= 8).getnnz(axis=1)
demo_users = user_ids[(favorite_counts >= 5) & (favorite_counts <= 10)][:3]
assert len(demo_users) == 3

for user_id in demo_users:
    row = user_book[user_ids.get_loc(user_id)]
    favorites = sorted(
        [(titles.iloc[i], rating) for i, rating in zip(row.indices, row.data) if rating >= 8],
        key=lambda item: (-item[1], item[0])
    )
    recommendations = recommend(user_id)
    print(f"User {user_id} | {len(favorites)} favorites")
    display(pd.DataFrame({
        "Highly rated book": pd.Series([f"{title} ({rating:g}/10)" for title, rating in favorites]),
        "Recommended book": pd.Series(recommendations),
    }).fillna(""))
    seen_titles = set(titles[lookup["ISBN"].isin(seen[user_id])].str.casefold())
    assert len(recommendations) == 10
    assert not seen_titles.intersection(title.casefold() for title in recommendations)

User 243 | 6 favorites


,Highly rated book,Recommended book
0,Memoirs Of A Geisha (10/10),Cradle And All
1,The Bean Trees (10/10),The Body Farm
2,Me Talk Pretty One Day (9/10),From Potter'S Field
3,The General'S Daughter (9/10),Animal Dreams
4,The Pilot'S Wife : A Novel (9/10),Point Of Origin
5,Unnatural Exposure (9/10),Tears Of The Giraffe (No.1 Ladies Detective Agency)
6,,Silent Prey
7,,The Book Of Ruth (Oprah'S Book Club (Paperback))
8,,Cruel & Unusual (Kay Scarpetta Mysteries (Paperback))
9,,Southern Cross


User 805 | 8 favorites


,Highly rated book,Recommended book
0,Where The Sidewalk Ends : Poems And Drawings (10/10),The Talented Mr. Ripley (Vintage Crime/Black Lizard)
1,Writing Down The Bones (10/10),Balzac And The Little Chinese Seamstress : A Novel
2,Interpreter Of Maladies (9/10),Bastard Out Of Carolina
3,Life Of Pi (9/10),Voyager
4,Dragonfly In Amber (8/10),Outlander
5,Girl With A Pearl Earring (8/10),Empire Falls
6,Jitterbug Perfume (8/10),Drums Of Autumn
7,She'S Come Undone (Oprah'S Book Club (Paperback)) (8/10),How The Garcia Girls Lost Their Accents (Plume Contemporary Fiction)
8,,"The Dilbert Principle: A Cubicle'S-Eye View Of Bosses, Meetings, Management Fads & Other Workplace Afflictions"
9,,One Hundred Years Of Solitude


User 1424 | 5 favorites


,Highly rated book,Recommended book
0,A Walk In The Woods: Rediscovering America On The Appalachian Trail (Official Guides To The Appalachian Trail) (8/10),Blackberry Wine
1,Memoirs Of A Geisha Uk (8/10),Naked Lunch
2,Plain Truth (8/10),On The Road
3,Prey (8/10),A Clockwork Orange (Norton Paperback Fiction)
4,The Name Of The Rose: Including Postscript To The Name Of The Rose (8/10),Atonement: A Novel
5,,The English Patient
6,,Stranger In A Strange Land (Remembering Tomorrow)
7,,Crow Lake (Today Show Book Club #7)
8,,Possession : A Romance
9,,The Hobbit: Or There And Back Again
